In [1]:
!pip install beautifulsoup4
!pip install glob
!pip install pandas


ERROR: Could not find a version that satisfies the requirement glob (from versions: none)
ERROR: No matching distribution found for glob


In [6]:
from bs4 import BeautifulSoup
import glob
import pandas as pd
import requests


key = "ea04f85bdd4b485c9c93f5faca413d05"
url = "https://newsapi.org/v2/top-headlines"  
cond = {"language": "en", "pageSize": 75, "apiKey": key}

call = requests.get(url, params=cond)
raw_art = call.json()
article_list = []
for a in raw_art.get("articles", []):
    raw_url = a.get("url")

    raw_text = None
    try:
        curr_art = requests.get(raw_url, timeout=15)
        s = BeautifulSoup(curr_art.text, "html.parser")

        paragraphs = [p.get_text() for p in s.find_all("p")]
        raw_text = " ".join(paragraphs)
    except Exception as e:
        raw_text = None
    article_list.append({
        "statement": raw_text
    })




df = pd.DataFrame(article_list)
df = df[df['statement'].notna() & (df['statement'].str.strip() != '')]

df


,statement
0,This live coverage has now ended - you can rea...
1,\n Quotes displayed in real-time or d...
3,Now 46 Tue 54 Wed 58 by KONSTANTIN TOROP...
6,Copyright 2025 The Associated Press. All Right...
7,"Watch CBS News \nOctober 27, 2025 / 12:05 PM E..."
9,Ahead of an expected global launch in the comi...
10,By \n\n The Associated Press\n \n \n ...
11,Guest Guest Login | Sign Up Push Square Guest...
12,Civilian activist Mouawia had just escaped el-...
13,POPULAR ARTICLES LATEST PODCASTS View All


In [7]:
df['statement'] = (
    df['statement']
    .astype(str)  
    .str.replace(r'\s+', ' ', regex=True) 
    .str.strip() 
)


In [8]:
df

,statement
0,This live coverage has now ended - you can rea...
1,Quotes displayed in real-time or delayed by at...
3,"Now 46 Tue 54 Wed 58 by KONSTANTIN TOROPIN, As..."
6,Copyright 2025 The Associated Press. All Right...
7,"Watch CBS News October 27, 2025 / 12:05 PM EDT..."
9,Ahead of an expected global launch in the comi...
10,By The Associated Press Catherine Connolly smi...
11,Guest Guest Login | Sign Up Push Square Guest ...
12,Civilian activist Mouawia had just escaped el-...
13,POPULAR ARTICLES LATEST PODCASTS View All
